In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
import re
from tqdm import tqdm
import logging

logging.getLogger("httpx").setLevel(logging.WARNING)
# Teacher Model Initialization
# Note: Ensure you have the necessary VRAM or use quantization (bitsandbytes) for 14B+ models.
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model_id = "Qwen/Qwen3-14B-AWQ" # 14B or 32B
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id,
            dtype=torch.float16, # torch.bfloat16 or torch.float16
            device_map=device) # device or "balanced"

Cross-Modal Distillation

In [ ]:
# only if device_map="balanced"
# device = model.device

SYSTEM_PROMPT = """You are a Home Assistant routing engine integrated with the GLaDOS persona.
You will receive a User Command, the user's detected Emotion, and the execution JSON payload.

You MUST output ONLY the verbal response from GLaDOS.
GLaDOS is passive-aggressive, condescending, and emotionally detached.
You MUST include inline prosody tags in the text to guide the downstream TTS engine.
Valid tags: <fast>, <slow_deadpan>, <pause>, <sigh>.
Do NOT output the JSON payload. Do NOT output markdown.

Example Output:
I have processed the request. <pause> Try to survive the darkness. <slow_deadpan> You're welcome.
"""

def generate_teacher_responses(user_command, user_emotion, json_payload, n_samples=3):
    """Generates N candidate responses for Self-Alignment Optimization."""
    prompt = f"User Emotion: {user_emotion}\nUser Command: {user_command}\nJSON Payload: {json_payload}\n"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(device)
    # Generate N diverse samples using sampling (do_sample=True)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        num_return_sequences=n_samples
    )
    return [tokenizer.decode(out[inputs.input_ids.shape[1]:], skip_special_tokens=True) for out in outputs]

In [ ]:
def evaluate_and_rank_candidate(candidate_text):
    """
    Evaluates a single SAO candidate.
    Returns a score (0 to 2) and the valid text.
    """
    score = 0
    raw_text = candidate_text.strip()
    # 1. Format Discipline (IFEval Strict Accuracy)
    # Fails if markdown backticks are detected
    if "```" in raw_text or "{" in raw_text:
        return 0, None
    score += 1
   # 2. Prosody Tag Adherence: Fails if no valid tags are embedded
    if re.search(r'<(fast|slow_deadpan|pause|sigh)>', raw_text):
        score += 1
    return score, raw_text

def apply_sao_selection(user_command, user_emotion, json_payload):
    """
    Self-Alignment Optimization: Generates candidates and selects the highest-ranked
    output based on format discipline and prosody inclusion.
    """
    candidates = generate_teacher_responses(user_command, user_emotion, json_payload, n_samples=3)
    best_candidate = None
    best_score = -1
    for candidate in candidates:
        score, valid_text = evaluate_and_rank_candidate(candidate)
        # Immediate short-circuit if a perfect label is generated
        if score == 2:
            return valid_text
        if score > best_score:
            best_score = score
            best_candidate = valid_text
    # Returns the highest-scored candidate (or None if all failed catastrophically)
    return best_candidate

In [ ]:
def create_ground_truth_dataset(input_csv, output_csv):
    df_input = pd.read_csv(input_csv)
    results = []
    # Counters for Evaluation Framework 1.3 metrics
    metrics = {
        "total_attempted": 0,
        "perfect_labels": 0,
        "failed_labels": 0
    }
    for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Distilling Teacher Knowledge"):
        user_cmd = row["User_Command"]
        user_emotion = row.get("Voice_Description", "neutral")
        # Ensure the column name matches the JSON payload in your current CSV structure
        json_payload = row.get("Assistant_Payload", "{}")
        metrics["total_attempted"] += 1
        valid_text = apply_sao_selection(user_cmd, user_emotion, json_payload)
        if valid_text is not None:
            results.append({
                "cmd_id": row.get("cmd_id", 0),
                "User_Command": user_cmd,
                "User_Emotion": user_emotion,
                "Target_JSON": json_payload,
                "Target_GLaDOS_Response": valid_text,
                "Audio_File": row.get("audio_file_name", "")
            })
            metrics["perfect_labels"] += 1
        else:
            metrics["failed_labels"] += 1

    df_output = pd.DataFrame(results)
    df_output.to_csv(output_csv, index=False)

    print("\n=== Evaluation 1.3 Results ===")
    print(f"Total Instances Processed: {metrics['total_attempted']}")
    print(f"Perfect Format & Prosody Conformance: {metrics['perfect_labels']} ({(metrics['perfect_labels']/metrics['total_attempted'])*100:.2f}%)")
    print(f"Discarded (Failed Strict Checks): {metrics['failed_labels']}")

In [ ]:
create_ground_truth_dataset("./data/description_prompts_train_with_audio.csv", "./data/multimodal_ground_truth_train.csv")

In [ ]:
create_ground_truth_dataset("./data/description_prompts_test_with_audio.csv", "./data/multimodal_ground_truth_test.csv")